# 선형회귀 실습

**Linear Regression · 최소제곱 · OLS**

입력의 선형 결합으로 출력을 예측하고 잔차 제곱합을 최소화하는 모델.

소재 분야에서 이해하기: 조성 비율로 격자 상수를 1차식으로 근사한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 선형 모델 문서](https://scikit-learn.org/stable/modules/linear_model.html)

## 1. 계수를 직접 읽어봅니다

선형회귀의 장점은 계수를 해석할 수 있다는 점입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
coefficients = model[-1].coef_
for name, value in zip(FEATURES, coefficients):
    print('%-8s 표준화 계수 %+.2f' % (name, value))
print('절편 %.1f HV' % model[-1].intercept_)
plt.bar(range(len(FEATURES)), coefficients)
plt.xticks(range(len(FEATURES)), ['temp', 'time', 'additive', 'noise'])
plt.ylabel('standardised coefficient'); plt.axhline(0, color='k', lw=1); plt.show()

## 2. 선형 모델의 한계

데이터에는 첨가비율의 제곱 항과 온도×시간 상호작용이 들어 있습니다. 항을 넣어주면 좋아집니다.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures

plain = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2').mean()
extended = cross_val_score(make_pipeline(PolynomialFeatures(2), LinearRegression()), X, y, cv=5, scoring='r2').mean()
print('1차 항만: R2 %.3f' % plain)
print('2차 항 포함: R2 %.3f' % extended)
print('\n계수 해석은 다른 변수를 고정했을 때의 변화량입니다. 변수들이 서로 상관되어 있으면 해석이 흔들립니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#linear-regression)을 여세요.